# Multilabel Classification Part 3

This notebook trains the same CNN experiment grid on multilabel image targets. The only required modeling change is a sigmoid output head with weighted binary cross-entropy instead of a softmax head with categorical cross-entropy.


## Optional Google Colab Bootstrap

For Colab Pro/A100, choose `Runtime > Change runtime type > A100 GPU`, then run from the top. The bootstrap cell is already configured to clone the repo into `/content` for faster training and mount Google Drive only for persistent outputs.

Leave the cell as-is for local Jupyter use; the Colab-only branch will not run outside Colab.


In [ ]:
# Colab/A100 run: do not force CPU.
# If you previously set CUDA_VISIBLE_DEVICES = "-1", restart the runtime before continuing.
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)


In [ ]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(f"Could not set memory growth for {gpu}: {exc}")

# Keep this experiment in float32 for now. This avoids the weighted-loss dtype mismatch.
mixed_precision.set_global_policy("float32")

print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
print("Mixed precision policy:", mixed_precision.global_policy())
if gpus:
    for index, gpu in enumerate(gpus):
        print(f"GPU {index} details:", tf.config.experimental.get_device_details(gpu))


In [ ]:
import os
import subprocess
from pathlib import Path

RUN_COLAB_BOOTSTRAP = True
GITHUB_REPO_URL = "https://github.com/xyldxal/hi-192-dental-xray.git"
USE_GOOGLE_DRIVE = True
COLAB_REPO_DIR = "/content/hi-192-dental-xray"

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and RUN_COLAB_BOOTSTRAP:
    if USE_GOOGLE_DRIVE:
        drive.mount("/content/drive")

    repo_root = Path(COLAB_REPO_DIR)
    repo_root.parent.mkdir(parents=True, exist_ok=True)

    if not (repo_root / ".git").exists():
        if not GITHUB_REPO_URL.strip():
            raise ValueError("Set GITHUB_REPO_URL before running the Colab bootstrap cell.")
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(repo_root)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], check=True)

    os.chdir(repo_root)
    print(f"Colab working directory set to: {repo_root}")
else:
    print(f"Current working directory: {Path.cwd()}")


In [ ]:
from pathlib import Path
import os
import sys

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "dental_opg_experiment.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. In Colab, run the bootstrap cell first or clone the repo manually."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import dental_opg_experiment as doe


In [ ]:
# Colab already includes a GPU TensorFlow build, so do not reinstall TensorFlow unless you must.
# If Colab raises an import error for image-classifiers/classification_models, run only this:
# %pip install image-classifiers

# For a brand-new local environment, use the full requirements file instead:
# %pip install -r requirements.txt


In [ ]:
DATASET_ROOT = None
MULTILABEL_SPLITS_DIR = PROJECT_ROOT / "artifacts" / "multilabel_splits"
MULTILABEL_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "multilabel_configs"

COLAB_OUTPUTS_DIR = Path("/content/drive/MyDrive/hi-192-dental-xray/outputs_multilabel")
if IN_COLAB and Path("/content/drive/MyDrive").exists():
    COLAB_OUTPUTS_DIR.parent.mkdir(parents=True, exist_ok=True)
    OUTPUTS_DIR = COLAB_OUTPUTS_DIR
else:
    OUTPUTS_DIR = PROJECT_ROOT / "outputs_multilabel"

required_split_files = [
    MULTILABEL_SPLITS_DIR / "train_manifest.csv",
    MULTILABEL_SPLITS_DIR / "val_manifest.csv",
    MULTILABEL_SPLITS_DIR / "test_manifest.csv",
]
required_config_file = MULTILABEL_CONFIGS_DIR / f"part_{PART_ID}_configs.csv" if "PART_ID" in globals() else MULTILABEL_CONFIGS_DIR / "part_3_configs.csv"

if not all(path.exists() for path in required_split_files) or not required_config_file.exists():
    print("Multilabel split/config artifacts not found; generating them now.")
    doe.write_multilabel_split_artifacts(dataset_root=DATASET_ROOT, out_dir=MULTILABEL_SPLITS_DIR, seed=42)
    doe.write_config_artifacts(out_dir=MULTILABEL_CONFIGS_DIR)

print(f"Dataset root: {doe.resolve_dataset_root(DATASET_ROOT)}")
print(f"Multilabel splits dir: {MULTILABEL_SPLITS_DIR}")
print(f"Multilabel configs dir: {MULTILABEL_CONFIGS_DIR}")
print(f"Outputs dir: {OUTPUTS_DIR}")


In [ ]:
PART_ID = 3
OVERWRITE_EXISTING = False
ONLY_CONFIG_IDS = []  # Example: ["base_cnn_drop_0p2_batch_16_dense_64"]
STOP_ON_ERROR = False
THRESHOLD = 0.5


In [ ]:
configs = doe.load_part_configs(PART_ID, configs_dir=MULTILABEL_CONFIGS_DIR)
print(f"Multilabel part {PART_ID} has {len(configs)} configs.")

for config in configs[:5]:
    print(config)


In [ ]:
import json

output_root = doe.resolve_outputs_dir(OUTPUTS_DIR) / f"part_{PART_ID}"
output_root.mkdir(parents=True, exist_ok=True)

selected_configs = [cfg for cfg in configs if not ONLY_CONFIG_IDS or cfg["config_id"] in ONLY_CONFIG_IDS]
results = []

for config in selected_configs:
    run_dir = output_root / config["config_id"]
    metrics_path = run_dir / "metrics.json"

    if metrics_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping completed run: {config['config_id']}")
        results.append(json.loads(metrics_path.read_text(encoding="utf-8")))
        continue

    print(
        f"Running {config['config_id']} -> {config['architecture']}, "
        f"dropout={config['dropout']}, batch={config['batch_size']}, dense={config['dense_units']}"
    )

    try:
        metrics = doe.train_single_multilabel_config(
            config=config,
            part_id=PART_ID,
            outputs_dir=OUTPUTS_DIR,
            splits_dir=MULTILABEL_SPLITS_DIR,
            dataset_root=DATASET_ROOT,
            threshold=THRESHOLD,
        )
        results.append(metrics)
    except Exception as exc:
        print(f"Failed on {config['config_id']}: {exc}")
        if STOP_ON_ERROR:
            raise

print(f"Collected {len(results)} result rows.")


In [ ]:
summary_df = doe.aggregate_part_results(outputs_dir=OUTPUTS_DIR, part_id=PART_ID)
summary_path = doe.resolve_outputs_dir(OUTPUTS_DIR) / f"part_{PART_ID}" / "part_summary.csv"

if not summary_df.empty:
    summary_df.to_csv(summary_path, index=False)
    display(summary_df.sort_values("f1_macro", ascending=False).head(10))
    print(f"Saved summary to: {summary_path}")
else:
    print("No completed runs yet.")
